# K-EmoCon Video Feature Extraction — frame-level Kaggle version

This version saves raw frame-level detections with timestamps as `P{pid}_frames.csv`, so local code can aggregate features over any shifted window, including the `t-400 ms` window for the annotation time `t`.

It also keeps legacy `P{pid}.csv` 5-second summaries for compatibility. The final archive is `/kaggle/working/video_features_cache.zip`.


In [ ]:

# Robust Kaggle runner for frame-level video features.
# It saves raw per-frame detections as P{pid}_frames.csv with timestamps,
# so local code can aggregate over shifted 400 ms windows, e.g. [t-0.400, t].
import os, sys, subprocess, textwrap, shutil
from pathlib import Path

WORK = Path('/kaggle/working') if Path('/kaggle').exists() else Path.cwd()
ENV_DIR = WORK / 'pyfeat_py310'
RUNNER = WORK / 'run_video_features.py'
RUNNER.write_text("\nimport os, re, tarfile, zipfile, warnings, traceback\nfrom pathlib import Path\nwarnings.filterwarnings('ignore')\n\nimport numpy as np\nimport pandas as pd\nimport cv2\n\n# We now save raw frame-level detections with timestamps.\n# Local aggregation can then use any shifted window, e.g. [t-0.400, t].\nFRAME_STRIDE = int(os.environ.get('FRAME_STRIDE', '1'))  # 1 = every frame\nSEGMENT_SEC = 5  # kept only for backward-compatible P{pid}.csv summaries\nAU_COLS = ['AU04', 'AU06', 'AU12', 'AU17']\nPOSE_COLS = ['Pitch', 'Yaw']\nRAW_COLS = AU_COLS + POSE_COLS\nOUT_NAMES = [f'vid_au{int(c[2:])}' for c in AU_COLS] + ['vid_pitch', 'vid_yaw']\nALL_COLS = AU_COLS + POSE_COLS\n\nWORK = Path('/kaggle/working') if Path('/kaggle').exists() else Path.cwd()\nCACHE_DIR = WORK / 'video_features_cache'\nCACHE_DIR.mkdir(parents=True, exist_ok=True)\nZIP_PATH = WORK / 'video_features_cache.zip'\nVIDEO_DIR = WORK / 'debate_recordings'\n\n\ndef find_input_root():\n    candidates = [\n        Path('/kaggle/input/kemocon-videos'),\n        Path('/kaggle/input/kemocon-videos/debate_recordings'),\n        Path('/kaggle/input/datasets/arinazamyshevskaya/kemocon-videos'),\n        Path('/kaggle/input/datasets/arinazamyshevskaya/kemocon-videos/debate_recordings'),\n        Path('/kaggle/input'),\n        Path.cwd(),\n    ]\n    for p in candidates:\n        if p.exists():\n            return p\n    return Path.cwd()\n\n\ndef safe_extract_tar(tar_path: Path, dest: Path):\n    print(f'Extracting {tar_path} -> {dest}', flush=True)\n    dest.mkdir(parents=True, exist_ok=True)\n    with tarfile.open(tar_path) as tf:\n        for member in tf.getmembers():\n            target = dest / member.name\n            if not str(target.resolve()).startswith(str(dest.resolve())):\n                raise RuntimeError(f'Unsafe tar member: {member.name}')\n        tf.extractall(dest)\n\n\ndef find_videos():\n    root = find_input_root()\n    if not list(VIDEO_DIR.rglob('p*.mp4')):\n        tar_candidates = []\n        for base in [root, Path('/kaggle/input') if Path('/kaggle/input').exists() else root]:\n            if base.exists():\n                tar_candidates += list(base.rglob('debate_recordings.tar'))\n                tar_candidates += list(base.rglob('*.tar'))\n                tar_candidates += list(base.rglob('*.tar.gz'))\n                tar_candidates += list(base.rglob('*.tgz'))\n        if tar_candidates:\n            safe_extract_tar(sorted(set(tar_candidates), key=lambda x: len(str(x)))[0], WORK)\n\n    search_roots = [VIDEO_DIR, root, Path('/kaggle/input') if Path('/kaggle/input').exists() else root]\n    vids = []\n    for base in search_roots:\n        if base.exists():\n            vids += list(base.rglob('p*.mp4'))\n    by_pid = {}\n    for v in sorted(set(vids)):\n        m = re.match(r'p(\\d+)_', v.name)\n        if not m:\n            continue\n        pid = int(m.group(1))\n        old = by_pid.get(pid)\n        if old is None or str(v).startswith(str(WORK)):\n            by_pid[pid] = v\n    return [by_pid[k] for k in sorted(by_pid)]\n\n\ndef video_meta(video_path: Path):\n    cap = cv2.VideoCapture(str(video_path))\n    fps = float(cap.get(cv2.CAP_PROP_FPS) or 30.0)\n    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)\n    width = float(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 1.0)\n    height = float(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 1.0)\n    cap.release()\n    return fps, n_frames, width, height\n\n\ndef base_frame_df(video_path: Path, pid: int, mode: str):\n    fps, n_frames, _, _ = video_meta(video_path)\n    frames = np.arange(0, max(n_frames, 0), max(1, FRAME_STRIDE), dtype=int)\n    df = pd.DataFrame({\n        'participant_id': pid,\n        'video_file': video_path.name,\n        'frame': frames,\n        'timestamp_sec': frames / fps,\n        'fps': fps,\n        'detection_mode': mode,\n    })\n    for col in RAW_COLS:\n        df[col] = np.nan\n    return df\n\n\ndef normalize_columns(df):\n    colmap = {}\n    lower = {str(c).lower(): c for c in df.columns}\n    variants = {\n        'AU04': ['au04', 'au4', 'au04_r', 'au04_c'],\n        'AU06': ['au06', 'au6', 'au06_r', 'au06_c'],\n        'AU12': ['au12', 'au12_r', 'au12_c'],\n        'AU17': ['au17', 'au17_r', 'au17_c'],\n        'Pitch': ['pitch', 'pose_pitch', 'headpose_pitch'],\n        'Yaw': ['yaw', 'pose_yaw', 'headpose_yaw'],\n        'frame': ['frame', 'frames', 'frame_id', 'input'],\n    }\n    for target, names in variants.items():\n        for name in names:\n            if name in lower:\n                colmap[lower[name]] = target\n                break\n    return df.rename(columns=colmap)\n\n\ndef coerce_frame_column(fex: pd.DataFrame, fps: float, n_frames: int):\n    fex = normalize_columns(fex.copy())\n\n    if 'frame' not in fex.columns:\n        # Some py-feat versions put timestamps or filenames in `input`.\n        for c in fex.columns:\n            vals = fex[c].astype(str)\n            extracted = vals.str.extract(r'(?:frame[_-]?)?(\\d+)')[0]\n            if extracted.notna().mean() > 0.8:\n                fex['frame'] = pd.to_numeric(extracted, errors='coerce')\n                break\n\n    if 'frame' not in fex.columns:\n        # Last resort: assume detections are in chronological order and map evenly.\n        if len(fex) == n_frames:\n            fex['frame'] = np.arange(n_frames)\n        else:\n            step = max(1, int(round(n_frames / max(1, len(fex)))))\n            fex['frame'] = np.arange(len(fex)) * step\n\n    fex['frame'] = pd.to_numeric(fex['frame'], errors='coerce')\n    fex = fex.dropna(subset=['frame'])\n    fex['frame'] = fex['frame'].round().astype(int)\n    fex = fex[(fex['frame'] >= 0) & (fex['frame'] < max(1, n_frames))]\n    fex['timestamp_sec'] = fex['frame'] / fps\n    return fex\n\n\ndef extract_frames_with_pyfeat(video_path: Path, pid: int, detector):\n    fps, n_frames, _, _ = video_meta(video_path)\n    print(f'  py-feat detect_video on every frame; fps={fps:.3f}, frames={n_frames}', flush=True)\n\n    # py-feat versions differ in whether skip_frames=0 is allowed.\n    try:\n        raw = detector.detect_video(str(video_path), skip_frames=max(0, FRAME_STRIDE - 1))\n    except TypeError:\n        raw = detector.detect_video(str(video_path))\n    except Exception:\n        # If skip_frames=0 fails, retry with skip_frames=1. Timestamps remain explicit.\n        if FRAME_STRIDE == 1:\n            raw = detector.detect_video(str(video_path), skip_frames=1)\n        else:\n            raise\n\n    if raw is None or len(raw) == 0:\n        return base_frame_df(video_path, pid, 'py-feat-empty')\n\n    fex = coerce_frame_column(pd.DataFrame(raw), fps, n_frames)\n    keep_extra = []\n    for c in fex.columns:\n        lc = str(c).lower()\n        if lc in ['faceboxes', 'facebox', 'confidence', 'input'] or 'confidence' in lc or 'face' in lc:\n            keep_extra.append(c)\n\n    out = pd.DataFrame({\n        'participant_id': pid,\n        'video_file': video_path.name,\n        'frame': fex['frame'].astype(int),\n        'timestamp_sec': fex['timestamp_sec'].astype(float),\n        'fps': fps,\n        'detection_mode': 'py-feat',\n    })\n    for col in RAW_COLS:\n        out[col] = pd.to_numeric(fex[col], errors='coerce') if col in fex.columns else np.nan\n    for col in keep_extra:\n        if col not in out.columns:\n            out[col] = fex[col].values\n    out = out.sort_values('frame').drop_duplicates('frame', keep='first').reset_index(drop=True)\n    return out\n\n\ndef extract_frames_fallback(video_path: Path, pid: int):\n    # Guaranteed fallback: frame-level rows + timestamps. AU columns are NaN.\n    # Pitch/Yaw are rough face-position proxies, not py-feat head pose.\n    fps, n_frames, width, height = video_meta(video_path)\n    cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'\n    face_cascade = cv2.CascadeClassifier(cascade_path)\n\n    rows = []\n    cap = cv2.VideoCapture(str(video_path))\n    frame_idx = 0\n    stride = max(1, FRAME_STRIDE)\n    while True:\n        ok, frame = cap.read()\n        if not ok:\n            break\n        if frame_idx % stride == 0:\n            row = {\n                'participant_id': pid,\n                'video_file': video_path.name,\n                'frame': frame_idx,\n                'timestamp_sec': frame_idx / fps,\n                'fps': fps,\n                'detection_mode': 'opencv-fallback',\n                'AU04': np.nan,\n                'AU06': np.nan,\n                'AU12': np.nan,\n                'AU17': np.nan,\n                'Pitch': np.nan,\n                'Yaw': np.nan,\n                'face_detected': 0,\n            }\n            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)\n            faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(40, 40))\n            if len(faces):\n                x, y, w, h = max(faces, key=lambda b: b[2] * b[3])\n                cx, cy = x + w / 2, y + h / 2\n                row['Yaw'] = float((cx - width / 2) / (width / 2) * 45.0)\n                row['Pitch'] = float((height / 2 - cy) / (height / 2) * 30.0)\n                row['face_detected'] = 1\n                row['face_x'] = int(x); row['face_y'] = int(y); row['face_w'] = int(w); row['face_h'] = int(h)\n            rows.append(row)\n        frame_idx += 1\n    cap.release()\n\n    if not rows:\n        return base_frame_df(video_path, pid, 'opencv-fallback')\n    return pd.DataFrame(rows)\n\n\ndef aggregate_5s_from_frames(frames_df: pd.DataFrame):\n    # Backward-compatible summary; local 400 ms aggregation should use *_frames.csv instead.\n    if len(frames_df) == 0:\n        return pd.DataFrame()\n    df = frames_df.copy()\n    df['segment_idx'] = np.floor(df['timestamp_sec'] / SEGMENT_SEC).astype(int)\n    rows = []\n    for seg_idx, seg in df.groupby('segment_idx', sort=True):\n        row = {'seconds': float((seg_idx + 1) * SEGMENT_SEC)}\n        for col, out in zip(ALL_COLS, OUT_NAMES):\n            if col in seg.columns:\n                vals = pd.to_numeric(seg[col], errors='coerce').dropna()\n                row[f'{out}_mean'] = float(vals.mean()) if len(vals) else np.nan\n                row[f'{out}_std'] = float(vals.std()) if len(vals) > 1 else np.nan\n            else:\n                row[f'{out}_mean'] = np.nan\n                row[f'{out}_std'] = np.nan\n        rows.append(row)\n    return pd.DataFrame(rows)\n\n\ndef main():\n    force_fallback = os.environ.get('FORCE_FALLBACK', '0') == '1'\n    mode = 'fallback'\n    detector = None\n    if not force_fallback:\n        try:\n            from feat import Detector\n            detector = Detector(au_model='svm')\n            mode = 'py-feat'\n            print('Detector ready: py-feat SVM mode', flush=True)\n        except Exception as e:\n            print('py-feat detector failed; using OpenCV fallback.', flush=True)\n            print(repr(e), flush=True)\n            mode = 'fallback'\n\n    videos = find_videos()\n    print(f'Found {len(videos)} videos: {[v.name for v in videos]}', flush=True)\n    if not videos:\n        raise FileNotFoundError('No p*.mp4 videos found. Check that the Kaggle dataset is attached.')\n\n    for vpath in videos:\n        m = re.match(r'p(\\d+)_', vpath.name)\n        pid = int(m.group(1)) if m else len(list(CACHE_DIR.glob('*_frames.csv'))) + 1\n        frames_cache = CACHE_DIR / f'P{pid}_frames.csv'\n        summary_cache = CACHE_DIR / f'P{pid}.csv'\n        if frames_cache.exists() and frames_cache.stat().st_size > 0:\n            print(f'P{pid}: frame cache exists, skipping.', flush=True)\n            continue\n\n        print(f'P{pid}: extracting frame-level detections from {vpath.name} [{mode}] ...', flush=True)\n        try:\n            if mode == 'py-feat' and detector is not None:\n                frames_df = extract_frames_with_pyfeat(vpath, pid, detector)\n            else:\n                frames_df = extract_frames_fallback(vpath, pid)\n        except Exception as e:\n            print(f'  ERROR while extracting P{pid}: {e}', flush=True)\n            print(traceback.format_exc(limit=2), flush=True)\n            frames_df = extract_frames_fallback(vpath, pid)\n\n        frames_df.to_csv(frames_cache, index=False)\n        summary_df = aggregate_5s_from_frames(frames_df)\n        summary_df.to_csv(summary_cache, index=False)\n\n        raw_nan = frames_df[RAW_COLS].isna().mean().mean() if set(RAW_COLS).issubset(frames_df.columns) else np.nan\n        print(f'  -> saved {frames_cache.name}: {len(frames_df)} rows, NaN(raw)={raw_nan:.1%}', flush=True)\n        print(f'  -> saved {summary_cache.name}: {len(summary_df)} legacy 5s segments', flush=True)\n\n    with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:\n        for f in sorted(CACHE_DIR.glob('*.csv')):\n            zf.write(f, f.name)\n    print(f'Saved {ZIP_PATH} ({ZIP_PATH.stat().st_size/1024:.0f} KB)', flush=True)\n    print('Download from: Kaggle -> Output -> video_features_cache.zip', flush=True)\n\nif __name__ == '__main__':\n    main()\n")


def run(cmd, **kwargs):
    print('>>>', ' '.join(map(str, cmd)), flush=True)
    return subprocess.run(list(map(str, cmd)), check=True, **kwargs)


def ensure_uv():
    try:
        import uv  # noqa: F401
    except Exception:
        run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])


def setup_py310_env():
    ensure_uv()
    run([sys.executable, '-m', 'uv', 'python', 'install', '3.10'])

    py = ENV_DIR / 'bin' / 'python'
    pip_marker = ENV_DIR / 'bin' / 'pip'
    if not py.exists() or not pip_marker.exists():
        if ENV_DIR.exists():
            shutil.rmtree(ENV_DIR)
        # --seed is important on Kaggle; otherwise the venv may not contain pip.
        run([sys.executable, '-m', 'uv', 'venv', '--seed', '--python', '3.10', ENV_DIR])

    py = ENV_DIR / 'bin' / 'python'
    marker = ENV_DIR / '.pyfeat_ready_frame_level_v2'
    if not marker.exists():
        run([py, '-m', 'pip', 'install', '-q', '-U', 'pip', 'setuptools<70', 'wheel'])
        run([py, '-m', 'pip', 'install', '-q', '--no-cache-dir',
             'numpy==1.26.4', 'pandas==2.2.2', 'matplotlib==3.8.4',
             'opencv-python-headless==4.10.0.84', 'scipy==1.13.1', 'scikit-learn==1.5.2',
             'scikit-image==0.24.0', 'pillow', 'imageio', 'h5py', 'joblib', 'natsort',
             'numexpr', 'PyWavelets', 'tqdm', 'requests', 'seaborn'])
        run([py, '-m', 'pip', 'install', '-q', '--no-cache-dir', 'py-feat==0.6.2'])
        marker.write_text('ok')
    return py

try:
    PY310 = setup_py310_env()
    run([PY310, RUNNER])
except Exception as e:
    print()
    print('Python 3.10 / py-feat path failed. Running guaranteed OpenCV fallback instead.')
    print('Reason:', repr(e))
    env = os.environ.copy()
    env['FORCE_FALLBACK'] = '1'
    run([sys.executable, RUNNER], env=env)


In [ ]:
# Optional quick check after the previous cell finishes
from pathlib import Path
import pandas as pd
cache = Path('/kaggle/working/video_features_cache')
frame_files = sorted(cache.glob('P*_frames.csv'))
summary_files = sorted(p for p in cache.glob('P*.csv') if not p.name.endswith('_frames.csv'))
print(f'Frame-level CSV files: {len(frame_files)}')
print(f'Legacy summary CSV files: {len(summary_files)}')
if frame_files:
    print(frame_files[0])
    display(pd.read_csv(frame_files[0]).head())
